In [1]:
import boto3
import time
import json 

In [2]:
client = boto3.client("cloudtrail", region_name="ap-southeast-1")

In [18]:
all_events = []
next_token = None
target_event_count = 500  # length of events

# collect until length of events hit desired number
while len(all_events) < target_event_count:
    # if subsequent pages of result (already have next token from first page), use next token
    if next_token:
        response = client.lookup_events(MaxResults=50, NextToken=next_token)
    # if first page of results, just get 50 rows
    else:
        response = client.lookup_events(MaxResults=50)
    
    events = response.get("Events", [])
    # extend list of all events since "events" is a list
    all_events.extend(events)

    # prevent getting rate limited by AWS API
    time.sleep(2)
    next_token = response.get("NextToken")
    if not next_token or len(events) == 0:
        break  # no more events available



In [4]:
print(f"Total events fetched: {len(all_events)}")

Total events fetched: 100


In [19]:
for i in all_events: 
    cloudtrail_event = json.loads(i["CloudTrailEvent"])
    if "errorMessage" in cloudtrail_event:
        # print(cloudtrail_event)
        print(cloudtrail_event["eventName"], cloudtrail_event["errorCode"], cloudtrail_event["errorMessage"], cloudtrail_event["eventSource"], cloudtrail_event["eventTime"])
        # if cloudtrail_event["userIdentity"]["userName"] == "s3cannot":
        # print(f"Error Code: {cloudtrail_event['errorCode']}\nError Message: {cloudtrail_event['errorMessage']}\n")     

ListResources GeneralServiceException AWS::GuardDuty::MalwareProtectionPlan Handler returned status FAILED: The AWS Access Key Id needs a subscription for the service (Service: GuardDuty, Status Code: 403, Request ID: 20b54362-bf4d-4b90-aa6c-b6254c58a5b9) (SDK Attempt Count: 1) (HandlerErrorCode: GeneralServiceException, RequestToken: 32fc9f14-b4ef-4ee3-b7ac-40ae08796939) cloudcontrolapi.amazonaws.com 2026-04-19T08:19:17Z
ListResources GeneralServiceException AWS::OpenSearchServerless::Collection Handler returned status FAILED: The AWS Access Key Id needs a subscription for the service (Service: OpenSearchServerless, Status Code: 400, Request ID: 23a314ef-2217-4db8-8afa-231cba001017) (SDK Attempt Count: 1) (HandlerErrorCode: GeneralServiceException, RequestToken: 8472c3e5-0fe6-4ab8-8eb9-82cd5b69e96f) cloudcontrolapi.amazonaws.com 2026-04-19T03:42:04Z
ListDomains InternalFailure An unknown error occurred codeartifact.amazonaws.com 2026-04-19T02:58:09Z
GetCatalog EntityNotFoundException 

In [ ]:
for i in all_events:
    for j in i["CloudTrailEvent"]:
        print(j)
    break